In [20]:
from sklearn.datasets import fetch_california_housing
import pandas as pd
import numpy as np
housing = fetch_california_housing(data_home=None, download_if_missing=True, return_X_y=False, as_frame=False, n_retries=3, delay=1.0)
housing
df = pd.DataFrame(data=housing.data, columns=housing.feature_names)
target = housing.target
df['target'] = target
df

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23,4.526
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22,3.585
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24,3.521
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25,3.413
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25,3.422
...,...,...,...,...,...,...,...,...,...
20635,1.5603,25.0,5.045455,1.133333,845.0,2.560606,39.48,-121.09,0.781
20636,2.5568,18.0,6.114035,1.315789,356.0,3.122807,39.49,-121.21,0.771
20637,1.7000,17.0,5.205543,1.120092,1007.0,2.325635,39.43,-121.22,0.923
20638,1.8672,18.0,5.329513,1.171920,741.0,2.123209,39.43,-121.32,0.847


In [21]:
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [22]:
n = len(df)
n_val = int(n*0.2)
n_test = int(n*0.2)
n_train = n - n_val - n_test

np.random.seed(2)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train+n_val]]
df_test = df.iloc[idx[n_train+n_val:]]
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)
print(len(df_train), len(df_val), len(df_test))
df_val.head(2)


12384 4128 4128


,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,target
0,8.1248,18.0,7.851309,1.02199,3189.0,3.339267,34.32,-118.52,3.740
1,2.2917,30.0,3.606762,1.08363,1425.0,2.535587,34.02,-118.48,3.308


In [23]:
y_train = np.log1p(df_train['target'].values)
y_val = np.log1p(df_val['target'].values)
y_test = np.log1p(df_test['target'].values)

del df_train['target']
del df_val['target']
del df_test['target']

In [24]:
def prepare_X(df):
    df = df.copy()
    features = housing.feature_names[0:6]
    X = df[features].values
    return X

In [25]:
def linear(X,y,r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones,X])

    XTX = X.T.dot(X)
    XTX = XTX + r*np.eye(XTX.shape[0])
    XTX_inv = np.linalg.inv(XTX)

    w_full = XTX_inv.dot(X.T).dot(y)
    return w_full[0], w_full[1:]

In [26]:
def rmse(y,y_pred):
    err = (y-y_pred)**2
    mse = err.mean()
    return np.sqrt(mse)

In [27]:
for v in [0, 0.00001, 0.0001, 0.001, 0.01, 0.1, 1, 10, 100,1000]:
    X_train = prepare_X(df_train)
    w0, w = linear(X_train, y_train, v)
    y_pred = w0 + X_train.dot(w)
    score = rmse(y_train, y_pred)
    print('%s: %s' % (v, score))

0: 0.24667046537863044
1e-05: 0.24667046537863044
0.0001: 0.24667046537863105
0.001: 0.24667046537869144
0.01: 0.24667046538472684
0.1: 0.24667046598801654
1: 0.24667052606786707
10: 0.2466762959692463
100: 0.24708888865731107
1000: 0.25513961477720415


In [28]:
r = 0.001

In [29]:
X_train = prepare_X(df_train)
w0, w = linear(X_train, y_train, r)
y_pred = w0 + X_train.dot(w)
score = rmse(y_train, y_pred)
score

np.float64(0.24667046537869144)

In [30]:
X_val = prepare_X(df_val)
y_pred = w0 + X_val.dot(w)
score = rmse(y_val, y_pred)
score

np.float64(0.24162592982129638)

In [31]:
df_full_train = pd.concat([df_train, df_val])
df_full_train = df_full_train.reset_index(drop=True)
X_full_train = prepare_X(df_full_train)
y_full_train = np.concatenate([y_train, y_val])
w0, w = linear(X_full_train, y_full_train, r)

In [32]:
X_test = prepare_X(df_test)
y_pred = w0 + X_test.dot(w)
score = rmse(y_test, y_pred)
score

np.float64(0.256462047536484)